# Visualización de curvas de luz ATLAS por clase

Notebook interactivo para explorar las simulaciones ATLAS y compararlas con ZTF/LSST.

**Estructura esperada de ficheros** (rutas relativas, ejecutar desde `simulations/`):
- Detecciones ZTF/LSST: `../data/simulated/checkpoints/det_*.parquet`
- Object info:          `../data/simulated/checkpoints/obj_*.parquet`
- Detecciones ATLAS:    `../data/simulated/atlas/checkpoints/det_atlas_*.parquet`
- Non-detecciones ATLAS (upper limits): `../data/simulated/atlas/checkpoints/ndet_atlas_*.parquet`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 110,
    'font.size': 10,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'legend.fontsize': 8,
})

## Configuración de rutas

In [ ]:
#Rutas relativas a simulations/, coherentes con generate.py / simulate_atlas_only.py
BASE_DIR  = Path('../data/simulated')
SIM_DIR   = BASE_DIR / 'checkpoints'            # detecciones ZTF + LSST + obj_info
ATLAS_DIR = BASE_DIR / 'atlas' / 'checkpoints'  # detecciones + non-detecciones ATLAS

# Clases disponibles
ALL_CLASSES = [
    'SNIa', 'SNII', 'SNIbc', 'SLSN',
    'QSO', 'AGN', 'Blazar', 'YSO', 'CV_Nova',
    'LPV', 'E', 'DSCT', 'RRL', 'CEP', 'Periodic-Other',
]

# Verificar rutas
for d in [SIM_DIR, ATLAS_DIR]:
    status = '✓' if d.exists() else 'NO EXISTE'
    print(f"{status}  {d}")

## Carga de datos

In [ ]:
def load_class_data(class_name: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Carga detecciones ZTF/LSST, ATLAS, non-detecciones ATLAS y object_info para una clase.
    Formato de ficheros ZTF/LSST:       det_{survey}_{class}_{index}.parquet
    Formato de ficheros ATLAS det:      det_atlas_{class}_{index}.parquet
    Formato de ficheros ATLAS non-det:  ndet_atlas_{class}_{index}.parquet
    Devuelve (det_sim, det_atlas, ndet_atlas, obj_info).
    """
    class_safe = class_name.replace('/', '_')

    # ZTF: det_ztf_{class}_*.parquet
    ztf_files = sorted(SIM_DIR.glob(f'det_ztf_{class_safe}*.parquet'))
    det_ztf   = pd.concat([pd.read_parquet(f) for f in ztf_files]) if ztf_files else pd.DataFrame()
    if not det_ztf.empty:
        det_ztf['tid'] = 'ZTF'

    # LSST: det_lsst_{class}_*.parquet
    lsst_files = sorted(SIM_DIR.glob(f'det_lsst_{class_safe}*.parquet'))
    det_lsst   = pd.concat([pd.read_parquet(f) for f in lsst_files]) if lsst_files else pd.DataFrame()
    if not det_lsst.empty:
        det_lsst['tid'] = 'LSST'

    # Combinar ZTF + LSST
    dfs = [df for df in [det_ztf, det_lsst] if not df.empty]
    det_sim = pd.concat(dfs) if dfs else pd.DataFrame()

    # ATLAS detecciones: det_atlas_{class}_*.parquet
    atlas_files = sorted(ATLAS_DIR.glob(f'det_atlas_{class_safe}*.parquet'))
    det_atlas   = pd.concat([pd.read_parquet(f) for f in atlas_files]) if atlas_files else pd.DataFrame()

    # ATLAS non-detecciones (upper limits): ndet_atlas_{class}_*.parquet
    ndet_files  = sorted(ATLAS_DIR.glob(f'ndet_atlas_{class_safe}*.parquet'))
    ndet_atlas  = pd.concat([pd.read_parquet(f) for f in ndet_files]) if ndet_files else pd.DataFrame()

    # Object info: obj_{class}_*.parquet
    obj_files = sorted(SIM_DIR.glob(f'obj_{class_safe}*.parquet'))
    obj_info  = pd.concat([pd.read_parquet(f) for f in obj_files]) if obj_files else pd.DataFrame()

    return det_sim, det_atlas, ndet_atlas, obj_info


# Precargar índice de oids por clase (para el selector interactivo)
print('Cargando índice de objetos...')
_oid_index = {}  # class_name -> list of oids con datos ATLAS
for cls in ALL_CLASSES:
    cls_safe = cls.replace('/', '_')
    atlas_files = sorted(ATLAS_DIR.glob(f'det_atlas_{cls_safe}*.parquet'))
    if atlas_files:
        df = pd.concat([pd.read_parquet(f) for f in atlas_files])
        _oid_index[cls] = sorted(df.index.unique().tolist())
    else:
        _oid_index[cls] = []
    print(f'  {cls:<20}: {len(_oid_index[cls])} objetos con ATLAS')

print('\nÍndice cargado.')

## Paletas de color por survey y banda

In [ ]:
# Colores por survey y banda
# ZTF: fid 1=g, 2=r, 3=i
ZTF_COLORS  = {1: '#2ecc71', 2: '#e74c3c', 3: '#8e44ad'}   # verde, rojo, morado
ZTF_LABELS  = {1: 'ZTF g',  2: 'ZTF r',  3: 'ZTF i'}

# LSST: fid 6=u, 1=g, 2=r, 3=i, 4=z, 5=y
LSST_COLORS = {
    6: '#1a1aff', 1: '#00cccc', 2: '#ff6600',
    3: '#cc0000', 4: '#660066', 5: '#330000'
}
LSST_LABELS = {6: 'LSST u', 1: 'LSST g', 2: 'LSST r',
               3: 'LSST i', 4: 'LSST z', 5: 'LSST y'}

# ATLAS: fid 0=c (cyan), 1=o (orange)
ATLAS_COLORS    = {0: '#00bcd4', 1: '#ff9800'}   # cyan, naranja
ATLAS_LABELS    = {0: 'ATLAS c', 1: 'ATLAS o'}
ATLAS_FID_NAMES = {0: 'c', 1: 'o'}               # para labels de upper limits

print("Paletas definidas.")

## Función de visualización

In [ ]:
def plot_lightcurve(
    oid:          str,
    det_sim:      pd.DataFrame,
    det_atlas:    pd.DataFrame,
    obj_info:     pd.DataFrame,
    ndet_atlas:   pd.DataFrame | None = None,
    show_ztf:     bool = True,
    show_lsst:    bool = True,
    show_atlas:   bool = True,
    show_uplims:  bool = True,
    mag_range:    tuple | None = None,
    ax:           plt.Axes | None = None,
) -> plt.Axes:
    """
    Dibuja la curva de luz completa de un objeto para los surveys seleccionados.
    Los upper limits ATLAS (ndet_atlas) se representan como flechas hacia abajo
    en el límite de detección (diffmaglim), distinguiendo la zona pre-explosión
    (MJD anterior a la primera detección ATLAS) con un fondo sombreado.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 4))

    # ZTF
    if show_ztf and not det_sim.empty:
        ztf_df = det_sim.loc[[oid]] if oid in det_sim.index else pd.DataFrame()
        ztf_df = ztf_df[ztf_df['tid'] == 'ZTF'] if 'tid' in ztf_df.columns else ztf_df
        for fid, color in ZTF_COLORS.items():
            band_df = ztf_df[ztf_df['fid'] == fid]
            if len(band_df) == 0:
                continue
            mag_col = 'magpsf_ml' if 'magpsf_ml' in band_df.columns else 'magpsf'
            sig_col = 'sigmapsf_ml' if 'sigmapsf_ml' in band_df.columns else 'sigmapsf'
            ax.errorbar(
                band_df['mjd'], band_df[mag_col], yerr=band_df[sig_col],
                fmt='o', color=color, markersize=3, alpha=0.7,
                elinewidth=0.5, capsize=1.5, label=ZTF_LABELS[fid],
            )

    #LSST
    if show_lsst and not det_sim.empty:
        lsst_df = det_sim.loc[[oid]] if oid in det_sim.index else pd.DataFrame()
        lsst_df = lsst_df[lsst_df['tid'] == 'LSST'] if 'tid' in lsst_df.columns else pd.DataFrame()
        for fid, color in LSST_COLORS.items():
            band_df = lsst_df[lsst_df['fid'] == fid]
            if len(band_df) == 0:
                continue
            mag_col = 'magpsf_ml' if 'magpsf_ml' in band_df.columns else 'magpsf'
            sig_col = 'sigmapsf_ml' if 'sigmapsf_ml' in band_df.columns else 'sigmapsf'
            ax.errorbar(
                band_df['mjd'], band_df[mag_col], yerr=band_df[sig_col],
                fmt='s', color=color, markersize=2.5, alpha=0.6,
                elinewidth=0.5, capsize=1.5, label=LSST_LABELS[fid],
            )

    #ATLAS
    first_atlas_mjd = None 
    if show_atlas and not det_atlas.empty:
        atlas_oid_df = det_atlas.loc[[oid]] if oid in det_atlas.index else pd.DataFrame()
        if not atlas_oid_df.empty:
            first_atlas_mjd = atlas_oid_df['mjd'].min()
        for fid, color in ATLAS_COLORS.items():
            band_df = atlas_oid_df[atlas_oid_df['fid'] == fid]
            if len(band_df) == 0:
                continue
            ax.errorbar(
                band_df['mjd'], band_df['magpsf'], yerr=band_df['sigmapsf'],
                fmt='^', color=color, markersize=4, alpha=0.85,
                elinewidth=0.7, capsize=2, label=ATLAS_LABELS[fid],
                markeredgewidth=0.5, markeredgecolor='k',
            )

    #ATLAS (no-detecciones
    if show_uplims and ndet_atlas is not None and not ndet_atlas.empty:
        ndet_oid_df = ndet_atlas.loc[[oid]] if oid in ndet_atlas.index else pd.DataFrame()
        if not ndet_oid_df.empty:
            pre_mask = (
                ndet_oid_df['mjd'] < first_atlas_mjd
                if first_atlas_mjd is not None
                else pd.Series(True, index=ndet_oid_df.index)
            )
            for fid, color in ATLAS_COLORS.items():
                band_df = ndet_oid_df[ndet_oid_df['fid'] == fid]
                if len(band_df) == 0:
                    continue
                band_pre  = band_df[band_df['mjd'] < first_atlas_mjd] if first_atlas_mjd is not None else band_df
                band_post = band_df[band_df['mjd'] >= first_atlas_mjd] if first_atlas_mjd is not None else pd.DataFrame()

                if len(band_pre):
                    ax.errorbar(
                        band_pre['mjd'], band_pre['diffmaglim'],
                        fmt='v', color=color, markersize=4, alpha=0.55,
                        label=f'ATLAS {ATLAS_FID_NAMES[fid]} uplim (pre-exp)',
                        markeredgewidth=0.3, markeredgecolor='k',
                    )
                if len(band_post):
                    ax.errorbar(
                        band_post['mjd'], band_post['diffmaglim'],
                        fmt='v', color=color, markersize=3, alpha=0.25,
                    )

            if first_atlas_mjd is not None:
                pre_mjds = ndet_oid_df[pre_mask]['mjd']
                if len(pre_mjds):
                    ax.axvspan(pre_mjds.min(), first_atlas_mjd,
                               alpha=0.06, color='gray', label='Ventana pre-exp')

    #Metadatos
    title = oid
    if not obj_info.empty and oid in obj_info.index:
        info = obj_info.loc[oid]
        extras = []
        for key in ['z', 't0', 'period', 'mean_mag_r']:
            if key in info.index and pd.notna(info[key]):
                extras.append(f"{key}={info[key]:.3f}")
        if extras:
            title += f"  ({', '.join(extras)})"

    ax.set_title(title, fontsize=9)
    ax.set_xlabel('MJD')
    ax.set_ylabel('Magnitud AB')
    ax.invert_yaxis()
    ax.legend(loc='upper right', ncol=3, framealpha=0.7)
    ax.grid(True, alpha=0.3)

    if mag_range:
        ax.set_ylim(mag_range[1], mag_range[0])  # invertido

    return ax


print("Función de visualización definida.")

## Panel de comparación por clase

Muestra N objetos aleatorios de una clase seleccionada.

In [ ]:
#Configuración
CLASS_TO_PLOT = 'SNIa'    # cambiar aquí
N_OBJECTS     = 4         # objetos a mostrar
RANDOM_SEED   = 42
SHOW_ZTF      = True
SHOW_LSST     = True
SHOW_ATLAS    = True

#Cargar datos
det_sim, det_atlas, ndet_atlas, obj_info = load_class_data(CLASS_TO_PLOT)

available_oids = _oid_index.get(CLASS_TO_PLOT, [])
if not available_oids:
    print(f"No hay objetos con detecciones ATLAS para la clase '{CLASS_TO_PLOT}'")
else:
    rng   = np.random.default_rng(RANDOM_SEED)
    oids  = rng.choice(available_oids, size=min(N_OBJECTS, len(available_oids)), replace=False)

    fig, axes = plt.subplots(len(oids), 1, figsize=(14, 4 * len(oids)))
    if len(oids) == 1:
        axes = [axes]

    fig.suptitle(f'Curvas de luz: {CLASS_TO_PLOT}', fontsize=13, y=1.01)

    for ax, oid in zip(axes, oids):
        plot_lightcurve(
            oid, det_sim, det_atlas, obj_info,
            ndet_atlas=ndet_atlas,
            show_ztf=SHOW_ZTF, show_lsst=SHOW_LSST, show_atlas=SHOW_ATLAS,
            ax=ax,
        )

    plt.tight_layout()
    plt.show()

## Panel multi-clase

Un objeto representativo de cada clase en un solo panel.

In [ ]:
#Configuración
CLASSES_TO_SHOW = ALL_CLASSES   # o una sublista, e.g. ['SNIa', 'QSO', 'RRL']
RANDOM_SEED     = 0
SHOW_ZTF        = True
SHOW_LSST       = True
SHOW_ATLAS      = True

rng = np.random.default_rng(RANDOM_SEED)

n_cols = 3
n_rows = int(np.ceil(len(CLASSES_TO_SHOW) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 3.5 * n_rows))
axes_flat = axes.flatten()

for i, cls in enumerate(CLASSES_TO_SHOW):
    ax = axes_flat[i]
    oids = _oid_index.get(cls, [])

    if not oids:
        ax.text(0.5, 0.5, f'Sin datos\n{cls}', ha='center', va='center',
                transform=ax.transAxes, fontsize=11)
        ax.set_title(cls)
        continue

    oid = rng.choice(oids)
    det_sim, det_atlas, ndet_atlas, obj_info = load_class_data(cls)

    plot_lightcurve(
        oid, det_sim, det_atlas, obj_info,
        ndet_atlas=ndet_atlas,
        show_ztf=SHOW_ZTF, show_lsst=SHOW_LSST, show_atlas=SHOW_ATLAS,
        ax=ax,
    )
    ax.set_title(f'{cls}   {oid}', fontsize=9)

# Ocultar ejes sobrantes
for j in range(len(CLASSES_TO_SHOW), len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle('Un objeto representativo por clase (ZTF, LSST y ATLAS)', fontsize=13, y=1.01)
plt.tight_layout()
# Para guardar la figura:
# fig.savefig(BASE_DIR / 'atlas' / 'simulations.pdf', bbox_inches='tight')
plt.show()

## Explorador interactivo por objeto

Escribe un `oid` específico para inspeccionarlo en detalle.

In [ ]:
#Selección manual
OID_TO_INSPECT = 'sim_SLSN_00803'   #cambiar aquí

# Inferir clase desde el oid
parts = OID_TO_INSPECT.split('_')
cls   = '_'.join(parts[1:-1]) if len(parts) >= 3 else None

if cls:
    det_sim, det_atlas, ndet_atlas, obj_info = load_class_data(cls)

    fig, axes = plt.subplots(2, 1, figsize=(14, 7))

    # Panel superior: todos los surveys
    plot_lightcurve(OID_TO_INSPECT, det_sim, det_atlas, obj_info,
                     ndet_atlas=ndet_atlas,
                     show_ztf=True, show_lsst=True, show_atlas=True, ax=axes[0])
    axes[0].set_title(f'{OID_TO_INSPECT} ZTF + LSST + ATLAS', fontsize=10)

    # Panel inferior: solo ATLAS
    plot_lightcurve(OID_TO_INSPECT, det_sim, det_atlas, obj_info,
                     ndet_atlas=ndet_atlas,
                     show_ztf=False, show_lsst=False, show_atlas=True, ax=axes[1])
    axes[1].set_title(f'{OID_TO_INSPECT} Solo ATLAS c/o', fontsize=10)

    plt.tight_layout()
    plt.show()

    # Estadísticas básicas
    if OID_TO_INSPECT in det_atlas.index:
        atlas_obj = det_atlas.loc[[OID_TO_INSPECT]]
        print(f"\nDetecciones ATLAS para {OID_TO_INSPECT}:")
        for fid, bname in [(0, 'c'), (1, 'o')]:
            band = atlas_obj[atlas_obj['fid'] == fid]
            if len(band):
                print(f"  Banda {bname}: {len(band)} detecciones, "
                      f"mag=[{band['magpsf'].min():.2f}, {band['magpsf'].max():.2f}], "
                      f"MJD=[{band['mjd'].min():.1f}, {band['mjd'].max():.1f}]")
else:
    print(f"No se pudo inferir la clase desde '{OID_TO_INSPECT}'")

## Comparación de profundidad por survey

Histograma de magnitudes de detección para verificar los límites de cada survey.

In [ ]:
#Configuración
CLASS_FOR_DEPTH = 'SNIa'

det_sim, det_atlas, _, _ = load_class_data(CLASS_FOR_DEPTH)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'Distribución de magnitudes de detección: {CLASS_FOR_DEPTH}', fontsize=12)

bins = np.linspace(14, 26, 60)

# ZTF
ax = axes[0]
ax.set_title('ZTF')
if not det_sim.empty:
    ztf_df = det_sim[det_sim['tid'] == 'ZTF'] if 'tid' in det_sim.columns else det_sim
    for fid, color in ZTF_COLORS.items():
        band = ztf_df[ztf_df['fid'] == fid]['magpsf'].dropna()
        band = band[band < 90]
        if len(band):
            ax.hist(band, bins=bins, color=color, alpha=0.6, label=ZTF_LABELS[fid])
ax.set_xlabel('Magnitud AB'); ax.set_ylabel('N detecciones'); ax.legend()
ax.axvline(20.5, color='k', ls='--', lw=1, label='Límite ZTF ~20.5')

# LSST
ax = axes[1]
ax.set_title('LSST')
bins_lsst = np.linspace(18, 28, 60)
if not det_sim.empty:
    lsst_df = det_sim[det_sim['tid'] == 'LSST'] if 'tid' in det_sim.columns else pd.DataFrame()
    for fid, color in LSST_COLORS.items():
        band = lsst_df[lsst_df['fid'] == fid]['magpsf'].dropna()
        band = band[band < 90]
        if len(band):
            ax.hist(band, bins=bins_lsst, color=color, alpha=0.6, label=LSST_LABELS[fid])
ax.set_xlabel('Magnitud AB'); ax.legend()
ax.axvline(24.5, color='k', ls='--', lw=1, label='Límite LSST ~24.5')

# ATLAS
ax = axes[2]
ax.set_title('ATLAS')
bins_atlas = np.linspace(14, 22, 40)
if not det_atlas.empty:
    for fid, color in ATLAS_COLORS.items():
        band = det_atlas[det_atlas['fid'] == fid]['magpsf'].dropna()
        band = band[band < 90]
        if len(band):
            ax.hist(band, bins=bins_atlas, color=color, alpha=0.7, label=ATLAS_LABELS[fid])
ax.set_xlabel('Magnitud AB'); ax.legend()
ax.axvline(19.5, color='k', ls='--', lw=1, label='Límite ATLAS ~19.5')

plt.tight_layout()
plt.show()

## Comparación cadencia entre surveys

In [ ]:
# Cadencia = gaps entre observaciones consecutivas por banda
CLASS_FOR_CADENCE = 'LPV'
N_OBJECTS_CAD     = 20   #promediar sobre N objetos

det_sim, det_atlas, _, _ = load_class_data(CLASS_FOR_CADENCE)
sample_oids = _oid_index.get(CLASS_FOR_CADENCE, [])[:N_OBJECTS_CAD]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(f'Distribución de cadencia: {CLASS_FOR_CADENCE} ({N_OBJECTS_CAD} objetos)', fontsize=11)

# ATLAS cadencia
ax = axes[0]
ax.set_title('ATLAS: gaps entre observaciones')
for fid, color, label in [(0, ATLAS_COLORS[0], 'c'), (1, ATLAS_COLORS[1], 'o')]:
    all_gaps = []
    if not det_atlas.empty:
        for oid in sample_oids:
            if oid not in det_atlas.index:
                continue
            band = det_atlas.loc[[oid]]
            band = band[band['fid'] == fid].sort_values('mjd')
            if len(band) > 1:
                gaps = np.diff(band['mjd'].values)
                all_gaps.extend(gaps[gaps < 30].tolist())
    if all_gaps:
        ax.hist(all_gaps, bins=50, color=color, alpha=0.7, label=f'ATLAS {label}')
ax.set_xlabel('Gap (días)'); ax.set_ylabel('N'); ax.legend()
ax.set_yscale('log')

# Número de detecciones por objeto por survey
ax = axes[1]
ax.set_title('N detecciones por objeto')

n_ztf, n_lsst, n_atlas = [], [], []
for oid in sample_oids:
    if not det_sim.empty and oid in det_sim.index:
        obj = det_sim.loc[[oid]]
        n_ztf.append(len(obj[obj['tid'] == 'ZTF']) if 'tid' in obj.columns else 0)
        n_lsst.append(len(obj[obj['tid'] == 'LSST']) if 'tid' in obj.columns else 0)
    if not det_atlas.empty and oid in det_atlas.index:
        n_atlas.append(len(det_atlas.loc[[oid]]))

bw = 0.25
x  = np.arange(len(sample_oids[:10]))
if n_ztf:   ax.bar(x - bw,   n_ztf[:10],   width=bw, label='ZTF',  color='#e74c3c', alpha=0.8)
if n_lsst:  ax.bar(x,        n_lsst[:10],  width=bw, label='LSST', color='#3498db', alpha=0.8)
if n_atlas: ax.bar(x + bw,   n_atlas[:10], width=bw, label='ATLAS',color='#ff9800', alpha=0.8)
ax.set_xlabel('Objeto (índice)'); ax.set_ylabel('N detecciones totales'); ax.legend()

plt.tight_layout()
plt.show()

## Objetos saltados en la simulación ATLAS

Objetos que tienen simulación ZTF/LSST pero fueron descartados por ATLAS
(no pasaron `_passes_preprocessor_filter`, menos de 6 detecciones).

In [ ]:
#Visualizar objetos saltados en la simulación ATLAS
# Los objetos saltados son los que están en obj_info (simulados en ZTF/LSST)
# pero no tienen detecciones ATLAS: no pasaron _passes_preprocessor_filter.

CLASS_SKIPPED = 'SLSN'   #cambiar aquí
N_SKIPPED     = 6        # número de objetos saltados a visualizar
RANDOM_SEED   = 42

det_sim, det_atlas, ndet_atlas, obj_info = load_class_data(CLASS_SKIPPED)

# OIDs con datos ZTF/LSST pero sin detecciones ATLAS
oids_with_atlas = set(det_atlas.index.unique()) if not det_atlas.empty else set()
oids_all        = set(obj_info.index.unique())  if not obj_info.empty  else set()
oids_skipped    = sorted(oids_all - oids_with_atlas)

print(f"Clase: {CLASS_SKIPPED}")
print(f"Objetos en obj_info (ZTF/LSST): {len(oids_all)}")
print(f"Objetos con detecciones ATLAS:   {len(oids_with_atlas)}")
print(f"Objetos saltados:                {len(oids_skipped)}")

if not oids_skipped:
    print("Ningún objeto saltado para esta clase.")
else:
    rng     = np.random.default_rng(RANDOM_SEED)
    sample  = list(rng.choice(oids_skipped,
                              size=min(N_SKIPPED, len(oids_skipped)),
                              replace=False))

    # Mostrar info de redshift y ventana temporal
    if not obj_info.empty:
        cols = [c for c in ['z', 'mjd_start', 'mjd_end', 't0'] if c in obj_info.columns]
        print(f"\nInfo de los objetos saltados (muestra de {len(sample)}):")
        print(obj_info.loc[sample, cols].to_string())
        if 'mjd_start' in obj_info.columns and 'mjd_end' in obj_info.columns:
            durations = obj_info.loc[sample, 'mjd_end'] - obj_info.loc[sample, 'mjd_start']
            print(f"\nDuración ventana (días): media={durations.mean():.0f}, "
                  f"min={durations.min():.0f}, max={durations.max():.0f}")

    # Plot: solo ZTF + LSST (sin ATLAS porque no hay)
    fig, axes = plt.subplots(len(sample), 1, figsize=(14, 4 * len(sample)))
    if len(sample) == 1:
        axes = [axes]
    fig.suptitle(f'Objetos saltados: {CLASS_SKIPPED} (sin detecciones ATLAS)',
                 fontsize=13, y=1.01)

    for ax, oid in zip(axes, sample):
        plot_lightcurve(
            oid, det_sim, det_atlas, obj_info,
            ndet_atlas=ndet_atlas,
            show_ztf=True, show_lsst=True, show_atlas=False,
            ax=ax,
        )
        # Anotar el redshift si hay
        if not obj_info.empty and oid in obj_info.index:
            info = obj_info.loc[oid]
            z = info.get('z', None)
            if z is not None and pd.notna(z):
                ax.set_title(f"{oid}  (z={z:.3f}) SIN ATLAS", fontsize=9, color='red')
            else:
                ax.set_title(f"{oid} SIN ATLAS", fontsize=9, color='red')

    plt.tight_layout()
    plt.show()

    # Comparar distribución de z entre saltados y detectados
    if not obj_info.empty and 'z' in obj_info.columns:
        fig, ax = plt.subplots(figsize=(8, 3.5))
        z_detected = obj_info.loc[obj_info.index.isin(oids_with_atlas), 'z'].dropna()
        z_skipped  = obj_info.loc[obj_info.index.isin(oids_skipped),    'z'].dropna()
        bins = np.linspace(0, max(z_detected.max(), z_skipped.max()) * 1.05, 40)
        ax.hist(z_detected, bins=bins, alpha=0.6, label=f'Con ATLAS (n={len(z_detected)})',
                color='#2ecc71', density=True)
        ax.hist(z_skipped,  bins=bins, alpha=0.6, label=f'Saltados  (n={len(z_skipped)})',
                color='#e74c3c', density=True)
        ax.set_xlabel('Redshift z')
        ax.set_ylabel('Densidad')
        ax.set_title(f'Distribución de z {CLASS_SKIPPED}: detectados vs saltados')
        ax.legend()
        plt.tight_layout()
        plt.show()

## Plegado en fase

Verificar consistencia de la periodicidad / eclipses plegando la curva al periodo conocido.

In [ ]:
#Plegado en fase para verificar consistencia de los eclipses
OID_TO_FOLD = 'sim_RRL_00782'   #cambiar aquí
# Inferir clase y cargar datos si no están ya cargados
parts = OID_TO_FOLD.split('_')
cls   = '_'.join(parts[1:-1]) if len(parts) >= 3 else None
det_sim, det_atlas, ndet_atlas, obj_info = load_class_data(cls)

# Obtener el periodo desde obj_info
period_col = next((c for c in ['period', 'Period', 'P'] if c in obj_info.columns), None)
if period_col is None or OID_TO_FOLD not in obj_info.index:
    print(f"No se encontró columna de periodo o el oid '{OID_TO_FOLD}' no está en obj_info.")
else:
    period = obj_info.loc[OID_TO_FOLD, period_col]
    print(f"Periodo usado: {period:.4f} días")

    fig, ax = plt.subplots(figsize=(10, 5))

    #ZTF + LSST
    if not det_sim.empty and OID_TO_FOLD in det_sim.index:
        obj_df = det_sim.loc[[OID_TO_FOLD]]
        for tid, colors, labels in [('ZTF', ZTF_COLORS, ZTF_LABELS),
                                     ('LSST', LSST_COLORS, LSST_LABELS)]:
            sub = obj_df[obj_df['tid'] == tid] if 'tid' in obj_df.columns else pd.DataFrame()
            for fid, color in colors.items():
                band = sub[sub['fid'] == fid]
                if len(band) == 0:
                    continue
                mag_col = 'magpsf_ml' if 'magpsf_ml' in band.columns else 'magpsf'
                phase = (band['mjd'] % period) / period
                ax.scatter(phase, band[mag_col], s=12, color=color,
                           alpha=0.7, label=labels[fid])

    #ATLAS
    if not det_atlas.empty and OID_TO_FOLD in det_atlas.index:
        atlas_obj = det_atlas.loc[[OID_TO_FOLD]]
        for fid, color in ATLAS_COLORS.items():
            band = atlas_obj[atlas_obj['fid'] == fid]
            if len(band) == 0:
                continue
            phase = (band['mjd'] % period) / period
            ax.scatter(phase, band['magpsf'], s=12, color=color,
                       alpha=0.7, marker='^', label=ATLAS_LABELS[fid])

    ax.invert_yaxis()  # convención astronómica: mag menor = más brillante, arriba
    ax.set_xlabel('Fase orbital (0–1)')
    ax.set_ylabel('Magnitud AB')
    ax.set_title(f'{OID_TO_FOLD}, curva plegada (P={period:.4f} d)')
    ax.legend(fontsize=7, ncol=3)
    plt.tight_layout()
    plt.show()